In [1]:
"""
PIPELINE STAGE: Capacity Dataset Enrichment, Temporal Standardization & Province Coding

INPUT: processed_data/steps/02_Capacity_Cleaned.xlsx
OUTPUT: processed_data/steps/03_Capacity_Enriched.xlsx
LIBRARIES: os, pandas

1. OBJECTIVE:
    Enrich the cleaned installed-capacity dataset by standardizing temporal attributes,
    correcting province name inconsistencies, assigning official Turkish province plate
    codes, and producing a chronologically ordered dataset suitable for integration with
    demographic, meteorological, generation, and environmental datasets.

2. TEMPORAL STANDARDIZATION:
    A. Month Name Conversion:
       Convert month names into numerical values for proper chronological ordering.

    B. Supported Formats:
       Accept both English and Turkish month representations, including
       abbreviated forms.

       Examples:

           January  → 1
           February → 2
           Dec      → 12
           Ocak     → 1
           Şubat    → 2
           Aralık   → 12

    C. Numeric Transformation:
       - Remove text-based month representations.
       - Convert all month values into integers (1–12).
       - Coerce invalid values to missing values when necessary.

3. PROVINCE NAME CORRECTION:
    A. Typographical Repair:
       Correct known province naming inconsistencies before coding.

       Examples:

           KÜTHAHYA → KÜTAHYA
           ELÂZIĞ   → ELAZIĞ

    B. Spatial Consistency:
       Ensure province names match the official naming convention
       used in the province coding reference table.

4. PROVINCE IDENTIFICATION:
    A. Dynamic Province Column Detection:
       Identify the province column by searching for:

           "iller"

       after lowercase normalization and Turkish character correction.

    B. Validation:
       - Stop processing if no province column is found.
       - Generate a warning message for structural inconsistencies.

5. OFFICIAL PROVINCE PLATE CODE ASSIGNMENT:
    A. Geographic Coding:
       Assign official Turkish province plate numbers based on
       province names.

    B. Province-to-Code Mapping:
       Match each province with its corresponding plate code.

       Examples:

           ADANA      → 1
           ANKARA     → 6
           İSTANBUL   → 34
           İZMİR      → 35
           KAHRAMANMARAŞ → 46

    C. Historical & Alternative Names:
       Support legacy or alternative province names when present.

       Examples:

           İÇEL       → 33
           ADAPAZARI  → 54
           URFA       → 63
           ANTAKYA    → 31

    D. Missing Code Detection:
       Report province names that cannot be matched to a valid
       plate code for manual review.

6. TEMPORAL DATA VALIDATION:
    A. Year Conversion:
       Convert the Year column to numeric format.

    B. Error Handling:
       Invalid year values should be converted to missing values
       rather than causing execution failure.

7. STRUCTURAL ENRICHMENT:
    A. New Variable Creation:
       Add:

           Plate

       as a new geographic identifier.

    B. Metadata Enhancement:
       Improve compatibility with external datasets that use
       plate numbers as the primary spatial key.

8. DATA SORTING & ORGANIZATION:
    A. Multi-Level Sorting:
       Sort all records in ascending order by:

           1. Year
           2. Month
           3. Plate

    B. Chronological Consistency:
       Ensure province-level observations follow a strict
       temporal sequence suitable for time-series analysis.

9. COLUMN REORDERING:
    A. Structural Layout:
       Position the newly created Plate column adjacent to
       the province identifier column.

    B. Dataset Readability:
       Improve dataset organization and facilitate downstream
       merging operations.

10. OUTPUT GENERATION:
    A. Export Destination:

           processed_data/steps/03_Capacity_Enriched.xlsx

    B. Export Settings:
       - Excel format (.xlsx)
       - No index column

11. PROCESS LOGGING & MONITORING:
    A. Runtime Messages:
       Display:
           - Source file name
           - Month conversion status
           - Province correction status
           - Plate code assignment status
           - Sorting status

    B. Warning Messages:
       Report:
           - Missing province column
           - Unmatched province names
           - Missing plate codes

    C. Completion Messages:
       Confirm successful creation and save location of the enriched dataset.

12. EXPECTED OUTPUT DATASET:
    The resulting dataset contains:

        - Province names
        - Official Plate codes
        - Year
        - Month (numeric)
        - Renewable installed-capacity indicators

    The dataset is fully standardized for temporal analysis,
    geographic integration, machine learning applications,
    clustering studies, and renewable energy sustainability research.
"""


import os
import pandas as pd

# 1. Dosya Yolları
BASE_DIR = r"C:\Users\w11\dergi2"
input_file = os.path.join(BASE_DIR, "processed_data", "steps", "02_capacity_cleaned.xlsx")
# Yeni dosyayı bir sonraki adım olarak kaydedelim
output_file = os.path.join(BASE_DIR, "processed_data", "steps", "03_capacity_enriched.xlsx")

# --- SÖZLÜKLER ---

# Ay isimlerini numaraya çevirme sözlüğü (Küçük harfle yazıldı, veriyi eşleştirirken lower() kullanacağız)
month_map = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
    "ocak": 1, "şubat": 2, "subat": 2, "mart": 3, "nisan": 4, "mayıs": 5, "mayis": 5, "haziran": 6,
    "temmuz": 7, "ağustos": 8, "agustos": 8, "eylül": 9, "eylul": 9, "ekim": 10, "kasım": 11, "kasim": 11, "aralık": 12, "aralik": 12
}

# 81 İlin Plaka Sözlüğü
plaka_map = {
    'ADANA': 1, 'ADIYAMAN': 2, 'AFYONKARAHİSAR': 3, 'AĞRI': 4, 'AMASYA': 5, 'ANKARA': 6, 'ANTALYA': 7, 'ARTVİN': 8, 'AYDIN': 9, 'BALIKESİR': 10,
    'BİLECİK': 11, 'BİNGÖL': 12, 'BİTLİS': 13, 'BOLU': 14, 'BURDUR': 15, 'BURSA': 16, 'ÇANAKKALE': 17, 'ÇANKIRI': 18, 'ÇORUM': 19, 'DENİZLİ': 20,
    'DİYARBAKIR': 21, 'EDİRNE': 22, 'ELAZIĞ': 23, 'ERZİNCAN': 24, 'ERZURUM': 25, 'ESKİŞEHİR': 26, 'GAZİANTEP': 27, 'GİRESUN': 28, 'GÜMÜŞHANE': 29, 'HAKKARİ': 30,
    'HATAY': 31, 'İSKENDERUN': 31, 'ANTAKYA': 31, 'ISPARTA': 32, 'MERSİN': 33, 'İÇEL': 33, 'İSTANBUL': 34, 'İZMİR': 35, 'KARS': 36, 'KASTAMONU': 37, 'KAYSERİ': 38,
    'KIRKLARELİ': 39, 'KIRŞEHİR': 40, 'KOCAELİ': 41, 'KONYA': 42, 'KÜTAHYA': 43, 'MALATYA': 44, 'MANİSA': 45, 'KAHRAMANMARAŞ': 46, 'MARDİN': 47, 'MUĞLA': 48,
    'MUŞ': 49, 'NEVŞEHİR': 50, 'NİĞDE': 51, 'ORDU': 52, 'RİZE': 53, 'SAKARYA': 54, 'ADAPAZARI': 54, 'SAMSUN': 55, 'SİİRT': 56, 'SİNOP': 57, 'SİVAS': 58, 'TEKİRDAĞ': 59,
    'TOKAT': 60, 'TRABZON': 61, 'TUNCELİ': 62, 'ŞANLIURFA': 63, 'URFA': 63, 'UŞAK': 64, 'VAN': 65, 'YOZGAT': 66, 'ZONGULDAK': 67, 'AKSARAY': 68, 'BAYBURT': 69,
    'KARAMAN': 70, 'KIRIKKALE': 71, 'BATMAN': 72, 'ŞIRNAK': 73, 'BARTIN': 74, 'ARDAHAN': 75, 'IĞDIR': 76, 'YALOVA': 77, 'KARABÜK': 78, 'KİLİS': 79,
    'OSMANİYE': 80, 'DÜZCE': 81
}

fix_map = {
    "KÜTHAHYA": "KÜTAHYA",
    "ELÂZIĞ": "ELAZIĞ"
}

def clean_province(text):
    if pd.isna(text):
        return text
   # typo fix
    text = fix_map.get(text, text)

    return text

def enrich_and_sort_data(input_path, output_path):
    print(f">>> İşleniyor: {os.path.basename(input_path)}")
    df = pd.read_excel(input_path)

    # Hangi sütunun iller olduğunu bul
    iller_col = None
    for col in df.columns:
        if "iller" in str(col).strip().lower().replace("i̇", "i"):
            iller_col = col
            break

    if not iller_col:
        print(" [!] Hata: 'İLLER' sütunu bulunamadı!")
        return

    # --- 1. Ay İsimlerini Numaraya Çevir ---
    if "Ay" in df.columns:
        # Önce string yap, küçük harfe çevirip boşlukları al, sonra sözlükten karşılığını bul
        df["Ay"] = df["Ay"].apply(lambda x: month_map.get(str(x).strip().lower(), x))
        # Nümerik değere zorla
        df["Ay"] = pd.to_numeric(df["Ay"], errors='coerce')
        df[iller_col] = df[iller_col].apply(clean_province)
        print(" [✓] 1. Aylar numaraya çevrildi.")
    else:
        print(" [!] 'Ay' sütunu bulunamadı.")

    # --- 3 & 4. Plaka Sütunu Ekle ve Değerleri Yazdır ---
    # İller sütunundaki değere bakıp plaka kodunu getiriyoruz
    df["Plaka"] = df[iller_col].map(plaka_map)
    print(" [✓] 3 & 4. Plaka sütunu eklendi ve kodlar dolduruldu.")

    # Plakası bulunamayan il var mı kontrol edelim (Yazım hatası vb. tespiti için)
    eksik_plakalar = df[df["Plaka"].isna()][iller_col].unique()
    if len(eksik_plakalar) > 0:
        print(f" [!] Uyarı: Plakası bulunamayan veriler var: {eksik_plakalar}")

    # --- 5. Yıl, Ay ve Plaka Bazında Artan Sıralama ---
    # Eğer Yıl sütunu string ise onu da sayıya çevirelim ki doğru sıralansın
    if "Yıl" in df.columns:
        df["Yıl"] = pd.to_numeric(df["Yıl"], errors='coerce')

    # Sıralamayı yap
    df = df.sort_values(by=["Yıl", "Ay", "Plaka"], ascending=[True, True, True])
    print(" [✓] 5. Veriler Yıl, Ay ve Plaka'ya göre artan (ascending) şekilde sıralandı.")

    # Sütun sırasını düzenlemek istersen (İsteğe bağlı, tablo daha şık dursun diye Plakayı başa alıyoruz)
    cols = list(df.columns)
    # Plakayı İller sütununun hemen yanına alalım
    if "Plaka" in cols:
        cols.insert(cols.index(iller_col), cols.pop(cols.index("Plaka")))
        df = df[cols]

    # Kaydet
    df.to_excel(output_path, index=False)
    print(f"\n✅ İŞLEM TAMAM! Yeni dosya kaydedildi: {output_path}")

# --- ÇALIŞTIR ---
if __name__ == "__main__":
    if os.path.exists(input_file):
        enrich_and_sort_data(input_file, output_file)
    else:
        print(f" [!] Dosya bulunamadı: {input_file}")

>>> İşleniyor: 02_capacity_cleaned.xlsx
 [✓] 1. Aylar numaraya çevrildi.
 [✓] 3 & 4. Plaka sütunu eklendi ve kodlar dolduruldu.
 [✓] 5. Veriler Yıl, Ay ve Plaka'ya göre artan (ascending) şekilde sıralandı.

✅ İŞLEM TAMAM! Yeni dosya kaydedildi: C:\Users\w11\dergi2\processed_data\steps\03_capacity_enriched.xlsx
